# AraBERT NER 


In [87]:
import csv
import hashlib
import inspect
import json
import math
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import numpy as np
import torch
import transformers
from transformers import (
    AutoModelForTokenClassification, AutoTokenizer, Trainer,
    TrainingArguments, default_data_collator, set_seed,
)

ROOT = Path.cwd().resolve()

if not (ROOT / "src" / "ner").is_dir():
    ROOT = ROOT.parent

if not (ROOT / "src" / "ner").is_dir():
    raise RuntimeError("Racine du projet introuvable.")
DATA = ROOT / "data" / "annotations" / "ner"
PATHS = {
    "train": DATA / "augmented" / "train_augmented.jsonl",
    "validation": DATA / "silver" / "validation.jsonl",
    "test": DATA / "silver" / "test.jsonl",
    "labels": DATA / "silver" / "labels.json",
}
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
ENTITY_TYPES = ("PER", "COURT", "ORG", "LOC")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| Transformers", transformers.__version__, "|", DEVICE)

PyTorch 2.11.0+cu128 | Transformers 5.23.0 | cuda


In [88]:
# Hyperparamètres de l'expérience précédente.
SEED = 42
MAX_LENGTH = 512
NUM_EPOCHS = 5
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

def require(condition, message):
    if not condition:
        raise ValueError(message)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def save_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
                    encoding="utf-8")

## 1. Vérification des données
Offsets en caractères Python, fin exclusive, sans normalisation des textes.
Les spans entities sont la référence. Une transition BIO ou un offset invalide
bloque l'exécution. Une différence entre la vue BIO et les spans est signalée ;
les labels d'entraînement sont reconstruits depuis les spans.
Aucun document n'est supprimé. Les identifiants et textes dupliqués sont refusés.

In [90]:
def load_jsonl(path):
    records = []
    with path.open(encoding="utf-8") as stream:
        for number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except ValueError as exc:
                raise ValueError(f"{path}:{number}: JSON invalide") from exc
            require(isinstance(row, dict), f"{path}:{number}: objet attendu")
            records.append(row)
    require(bool(records), f"{path}: corpus vide")
    return records

labels = json.loads(PATHS["labels"].read_text(encoding="utf-8"))
expected_labels = {"O"} | {f"{p}-{t}" for t in ENTITY_TYPES for p in ("B", "I")}
require(isinstance(labels, list) and all(isinstance(x, str) for x in labels),
        "labels.json doit être une liste de chaînes")
require(len(labels) == len(expected_labels) and set(labels) == expected_labels,
        "Inventaire attendu : O et B/I pour PER, COURT, ORG, LOC")
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
print(label2id)

{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-COURT': 3, 'I-COURT': 4, 'B-ORG': 5, 'I-ORG': 6, 'B-LOC': 7, 'I-LOC': 8}


In [91]:
def bio_spans(tags, strict=True):
    """Spans de tokens ; aucun I isolé n'est transformé en B."""
    spans, active = [], None
    for i, tag in enumerate([*tags, "O"]):
        require(isinstance(tag, str) and tag in label2id, f"Label inconnu : {tag}")
        prefix, kind = ("O", None) if tag == "O" else tag.split("-", 1)
        continues = prefix == "I" and active is not None and active[0] == kind
        if active is not None and not continues:
            spans.append((active[0], active[1], i))
            active = None
        if prefix == "B":
            active = (kind, i)
        elif prefix == "I" and not continues:
            require(not strict, f"Transition BIO invalide au token {i}: {tag}")
    return spans

def validate_offsets(text, rows, text_key):
    previous_end = 0
    for row in rows:
        require(isinstance(row, dict), "Annotation non objet")
        start, end = row.get("start"), row.get("end")
        require(type(start) is int and type(end) is int, "Offsets non entiers")
        require(previous_end <= start < end <= len(text), "Offsets invalides/chevauchants")
        require(row.get(text_key) == text[start:end], "Texte différent des offsets")
        previous_end = end

In [92]:
def audit_records(records, split, seen_ids, seen_texts):
    counts, view_mismatches = Counter(), []
    for row in records:
        identifier = row.get("id")
        try:
            for field in ("id", "text", "legal_class"):
                require(isinstance(row.get(field), str) and bool(row[field].strip()),
                        f"Champ invalide : {field}")
            require(row.get("split") == split, "Split déclaré incorrect")
            text = row["text"]
            fingerprint = hashlib.sha256(text.encode("utf-8")).hexdigest()
            require(identifier not in seen_ids, f"Identifiant dupliqué : {identifier}")
            require(fingerprint not in seen_texts, "Texte dupliqué dans/entre les splits")
            if "sha256" in row:
                require(row["sha256"] == fingerprint, "Empreinte du texte incorrecte")
            entities, tokens = row.get("entities"), row.get("tokens")
            require(isinstance(entities, list), "entities doit être une liste")
            require(isinstance(tokens, list) and bool(tokens), "tokens doit être non vide")
            validate_offsets(text, entities, "text")
            validate_offsets(text, tokens, "token")
            for entity in entities:
                require(entity.get("label") in ENTITY_TYPES, "Type d'entité inconnu")
                require(entity["text"].strip() == entity["text"] and entity["text"],
                        "Entité vide ou espaces périphériques")
                counts[entity["label"]] += 1
            previous_end = 0
            for token in tokens:
                require(not text[previous_end:token["start"]].strip(), "Texte non couvert")
                previous_end = token["end"]
            require(not text[previous_end:].strip(), "Fin du texte non couverte")
            decoded = [(kind, tokens[a]["start"], tokens[b-1]["end"])
                       for kind, a, b in bio_spans([t.get("tag") for t in tokens])]
            stored = [(e["label"], e["start"], e["end"]) for e in entities]
            if decoded != stored:
                view_mismatches.append(identifier)
            seen_ids[identifier], seen_texts[fingerprint] = split, split
        except (ValueError, TypeError, KeyError) as exc:
            raise ValueError(f"{split}/{identifier}: {exc}") from exc
    return {"split": split, "documents": len(records),
            "entities": {kind: counts[kind] for kind in ENTITY_TYPES},
            "bio_view_mismatches": view_mismatches}

def show_audit(report):
    print(report["split"], "| documents:", report["documents"], "|", report["entities"])
    print("Écarts BIO/spans:", len(report["bio_view_mismatches"]))

In [93]:
# Test n'est pas lu ici.
records = {split: load_jsonl(PATHS[split]) for split in ("train", "validation")}
seen_ids, seen_texts, audits = {}, {}, {}
for split in ("train", "validation"):
    audits[split] = audit_records(records[split], split, seen_ids, seen_texts)
    show_audit(audits[split])
source_hashes = {split: sha256(PATHS[split]) for split in ("train", "validation", "labels")}

train | documents: 290 | {'PER': 777, 'COURT': 586, 'ORG': 382, 'LOC': 405}
Écarts BIO/spans: 40
validation | documents: 10 | {'PER': 55, 'COURT': 80, 'ORG': 32, 'LOC': 5}
Écarts BIO/spans: 9


## 2. Tokenisation et alignement BIO
Tokenizer rapide AraBERT, sans troncature ni prétraitement externe.
Pour préserver exactement les spans (par exemple Muhammad après un préfixe arabe
accolé), les frontières d'annotation délimitent les fragments à tokeniser.
Leurs tokens sont réunis dans l'ordre du document ; le modèle reçoit ensuite
les fenêtres complètes, pas des entités isolées. Les offsets locaux sont
replacés dans le texte original, qui reste inchangé.

Ce protocole utilise les frontières annotées pour la préparation des trois jeux ;
les scores mesurent la classification dans ce protocole, pas une segmentation
entièrement automatique de texte brut. Tout span non reconstructible est refusé.

In [94]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
require(tokenizer.is_fast, "Tokenizer rapide obligatoire")
require(all(v is not None for v in (
    tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id)),
    "Tokens spéciaux BERT manquants")
require(3 <= MAX_LENGTH <= tokenizer.model_max_length, "Longueur incompatible")
require(tokenizer.num_special_tokens_to_add(pair=False) == 2,
        "Format attendu : [CLS] texte [SEP]")
print(type(tokenizer).__name__, "| longueur maximale:", MAX_LENGTH)

BertTokenizer | longueur maximale: 512


In [95]:
def encode_with_boundaries(row):
    """Tokenise tous les fragments et conserve les offsets du document."""
    boundaries = sorted({0, len(row["text"])} | {
        position for entity in row["entities"]
        for position in (entity["start"], entity["end"])
    })
    ids, offsets = [], []
    for start, end in zip(boundaries, boundaries[1:]):
        encoded = tokenizer(
            row["text"][start:end], add_special_tokens=False, truncation=False,
            return_offsets_mapping=True, return_attention_mask=False,
            return_token_type_ids=False,
        )
        require(len(encoded["input_ids"]) == len(encoded["offset_mapping"]),
                "Nombre d'offsets différent du nombre de tokens")
        ids.extend(encoded["input_ids"])
        offsets.extend((start + a, start + b) for a, b in encoded["offset_mapping"])
    require(bool(ids), "Document sans sous-token")
    return ids, offsets

In [96]:
def tokenize_document(row):
    ids, offsets = encode_with_boundaries(row)
    entities = row["entities"]
    tags, owners = [], []
    entity_index, previous_end = 0, 0
    for start, end in offsets:
        require(previous_end <= start < end <= len(row["text"]), "Offset tokenizer invalide")
        previous_end = end
        while entity_index < len(entities) and entities[entity_index]["end"] <= start:
            entity_index += 1
        owner, tag = None, "O"
        if entity_index < len(entities) and entities[entity_index]["start"] < end:
            entity = entities[entity_index]
            require(entity["start"] <= start and end <= entity["end"],
                    f"Sous-token [{start}, {end}) traverse une frontière d'entité")
            owner = entity_index
            prefix = "I" if owners and owners[-1] == owner else "B"
            tag = f"{prefix}-{entity['label']}"
        tags.append(tag)
        owners.append(owner)
    decoded = [(kind, offsets[a][0], offsets[b-1][1]) for kind, a, b in bio_spans(tags)]
    expected = [(e["label"], e["start"], e["end"]) for e in entities]
    require(decoded == expected, "Entité perdue ou frontières non reconstructibles exactement")
    return ids, offsets, [label2id[tag] for tag in tags], owners

## 3. Documents longs
Fenêtres de 510 sous-tokens utiles au maximum, plus CLS et SEP.
Le découpage recule avant une entité traversant la frontière : aucune entité n'est
coupée et aucun sous-token n'est perdu ou dupliqué. Une entité seule dépassant la
capacité est refusée explicitement. Les tokens spéciaux et le padding portent -100.
Sans chevauchement, chaque entité contribue une seule fois aux métriques.

In [97]:
def make_windows(row):
    ids, offsets, targets, owners = tokenize_document(row)
    capacity = MAX_LENGTH - 2
    windows, start = [], 0
    while start < len(ids):
        end = min(start + capacity, len(ids))
        if end < len(ids) and owners[end] is not None and owners[end-1] == owners[end]:
            owner = owners[end]
            while end > start and owners[end-1] == owner:
                end -= 1
            require(end > start, f"Entité plus longue qu'une fenêtre de {capacity} sous-tokens")
        content = [tokenizer.cls_token_id, *ids[start:end], tokenizer.sep_token_id]
        padding = MAX_LENGTH - len(content)
        window = {
            "input_ids": content + [tokenizer.pad_token_id] * padding,
            "attention_mask": [1] * len(content) + [0] * padding,
            "labels": [-100, *targets[start:end], -100] + [-100] * padding,
        }
        if "token_type_ids" in tokenizer.model_input_names:
            window["token_type_ids"] = [0] * MAX_LENGTH
        windows.append(window)
        start = end
    require(sum(sum(v != -100 for v in w["labels"]) for w in windows) == len(ids),
            "Perte ou duplication de sous-tokens")
    return windows

def prepare_windows(rows, split):
    windows = []
    for row in rows:
        try:
            windows.extend(make_windows(row))
        except (ValueError, TypeError) as exc:
            raise ValueError(f"{split}/{row['id']}: {exc}") from exc
    require(bool(windows), f"{split}: aucune fenêtre")
    print(split, "| documents:", len(rows), "| fenêtres:", len(windows))
    return windows

In [98]:
model_data = {split: prepare_windows(records[split], split)
              for split in ("train", "validation")}

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (517 > 512). Running this sequence through the model will result in indexing errors


train | documents: 290 | fenêtres: 371
validation | documents: 10 | fenêtres: 31


## 4. Métriques d'entités
Comparaison exacte du type et des deux frontières en BIO strict.
Les I prédits sans B compatible sont ignorés, jamais réparés en B.
Macro-F1 : moyenne non pondérée des F1 des quatre types, zéro pour un type absent.
Les scores micro sont aussi rapportés ; seul Macro-F1 sélectionne le modèle.

In [101]:
def entity_report(eval_prediction):
    logits, references = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    logits, references = np.asarray(logits), np.asarray(references)
    require(logits.ndim == 3 and logits.shape[:2] == references.shape
            and logits.shape[-1] == len(labels), "Dimensions de prédiction incohérentes")
    require(np.isfinite(logits).all(), "Logits non finis")
    predictions = logits.argmax(axis=-1)
    tp, fp, fn = Counter(), Counter(), Counter()
    correct, total = 0, 0
    for predicted, reference in zip(predictions, references):
        mask = reference != -100
        gold = [id2label[int(v)] for v in reference[mask]]
        guessed = [id2label[int(v)] for v in predicted[mask]]
        expected = set(bio_spans(gold))
        actual = set(bio_spans(guessed, strict=False))
        tp.update(kind for kind, _, _ in expected & actual)
        fp.update(kind for kind, _, _ in actual - expected)
        fn.update(kind for kind, _, _ in expected - actual)
        correct += int((predicted[mask] == reference[mask]).sum())
        total += int(mask.sum())
    require(total > 0, "Aucun token évaluable")
    def scores(t, p, n):
        return {
            "precision": t / (t+p) if t+p else 0.0,
            "recall": t / (t+n) if t+n else 0.0,
            "f1": 2*t / (2*t+p+n) if 2*t+p+n else 0.0,
            "support": t+n,
        }
    per_entity = {kind: scores(tp[kind], fp[kind], fn[kind]) for kind in ENTITY_TYPES}
    macro = {key: sum(row[key] for row in per_entity.values()) / len(ENTITY_TYPES)
             for key in ("precision", "recall", "f1")}
    return {"per_entity": per_entity, "macro": macro,
            "micro": scores(sum(tp.values()), sum(fp.values()), sum(fn.values())),
            "token_accuracy": correct / total, "evaluated_tokens": total}

In [102]:
def compute_metrics(prediction):
    report = entity_report(prediction)
    return {"macro_f1": report["macro"]["f1"], "micro_f1": report["micro"]["f1"],
            "precision": report["micro"]["precision"], "recall": report["micro"]["recall"],
            "token_accuracy": report["token_accuracy"]}

def show_report(report, title):
    print(title)
    print(f"{'Entité':<10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>9}")
    for kind, row in report["per_entity"].items():
        print(f"{kind:<10} {row['precision']:>10.4f} {row['recall']:>10.4f} "
              f"{row['f1']:>10.4f} {row['support']:>9}")
    print("Macro-F1:", round(report["macro"]["f1"], 4),
          "| Micro-F1:", round(report["micro"]["f1"], 4))

In [103]:
# Trois PER corrects, les trois autres types manqués.
gold = np.array([[-100, label2id["B-PER"], label2id["B-PER"], label2id["B-PER"],
                  label2id["B-COURT"], label2id["B-ORG"], label2id["B-LOC"], -100]])
guessed = np.array([[0, label2id["B-PER"], label2id["B-PER"], label2id["B-PER"], 0, 0, 0, 0]])
check = entity_report((np.eye(len(labels))[guessed], gold))
require(math.isclose(check["macro"]["f1"], 0.25), "Erreur Macro-F1")
require(math.isclose(check["micro"]["f1"], 2/3), "Erreur Micro-F1")
require(bio_spans(["I-PER"], strict=False) == [], "Un I isolé ne doit pas être réparé")
print("Contrôles des métriques : OK")

Contrôles des métriques : OK


## 5. Fine-tuning depuis AraBERT initial
La tête de classification est initialisée pour les neuf labels BIO.
Tous les paramètres sont entraînés ; CUDA si disponible, sinon CPU.
Évaluation et sauvegarde à chaque époque, sélection exclusivement sur Validation.
Chaque exécution de la cellule suivante crée un dossier inédit.

In [104]:
set_seed(SEED)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id,
)
require(model.config.model_type == "bert", "Architecture BERT attendue")
require(MAX_LENGTH <= model.config.max_position_embeddings, "Longueur hors capacité")
model.to(DEVICE)
RUN_ID = "augmented-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
RUN_DIR = ROOT / "artifacts" / "ner" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Nouvelle expérience :", RUN_DIR)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 497.28it/s]
[transformers] BertForTokenClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arc

Nouvelle expérience : C:\Users\PULSE\Documents\Master(MLAIM)\Projets personnels\ArabicLegalTextMining\artifacts\ner\augmented-20261120T212928Z-abcb8b06


In [105]:
training_options = {
    "output_dir": str(RUN_DIR),
    "num_train_epochs": NUM_EPOCHS, "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "weight_decay": WEIGHT_DECAY,
    "save_strategy": "epoch", "logging_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1", "greater_is_better": True,
    "save_total_limit": 2, "seed": SEED, "data_seed": SEED,
    "dataloader_num_workers": 0, "report_to": [], "push_to_hub": False,
    "fp16": False, "bf16": False,
}
argument_names = inspect.signature(TrainingArguments).parameters
eval_key = "eval_strategy" if "eval_strategy" in argument_names else "evaluation_strategy"
training_options[eval_key] = "epoch"
cpu_key = "use_cpu" if "use_cpu" in argument_names else "no_cuda"
training_options[cpu_key] = DEVICE == "cpu"
if "warmup_ratio" in argument_names:
    training_options["warmup_ratio"] = WARMUP_RATIO
else:
    # Transformers 5.23 : warmup_steps accepte une fraction entre 0 et 1.
    require("warmup_steps" in argument_names, "API warmup non prise en charge")
    training_options["warmup_steps"] = WARMUP_RATIO
training_args = TrainingArguments(**training_options)
require(training_args.device.type == DEVICE, "Périphérique Trainer différent du choix CUDA/CPU")

In [106]:
trainer_options = {
    "model": model, "args": training_args,
    "train_dataset": model_data["train"], "eval_dataset": model_data["validation"],
    "data_collator": default_data_collator, "compute_metrics": compute_metrics,
}
trainer_names = inspect.signature(Trainer).parameters
processor_key = "processing_class" if "processing_class" in trainer_names else "tokenizer"
trainer_options[processor_key] = tokenizer
trainer = Trainer(**trainer_options)
parameters = {
    "base_model": MODEL_NAME, "seed": SEED, "max_length": MAX_LENGTH,
    "entity_types": list(ENTITY_TYPES), "label2id": label2id,
    "selection_split": "validation", "selection_metric": "eval_macro_f1",
    "macro_includes_absent_types": True, "warmup_ratio": WARMUP_RATIO,
    "training_arguments": training_args.to_dict(),
    "data_paths": {split: str(PATHS[split]) for split in ("train", "validation", "labels")},
    "data_sha256": source_hashes, "device": DEVICE,
    "versions": {"torch": str(torch.__version__), "transformers": transformers.__version__},
    "window_counts": {split: len(rows) for split, rows in model_data.items()},
    "test_used_for_selection": False,
    "tokenization": "annotation_boundary_fragments_with_global_offsets",
}
save_json(RUN_DIR / "parameters.json", parameters)
save_json(RUN_DIR / "training_data_audit.json", audits)

In [107]:
train_result = trainer.train(resume_from_checkpoint=False)
best_checkpoint = trainer.state.best_model_checkpoint
require(best_checkpoint is not None and trainer.state.best_metric is not None,
        "Aucun checkpoint sélectionné sur Validation")
require(Path(best_checkpoint).resolve().is_relative_to(RUN_DIR.resolve()),
        "Checkpoint sélectionné hors de la nouvelle expérience")
# load_best_model_at_end a rechargé le meilleur modèle.
print("Meilleur checkpoint :", best_checkpoint)
print("Macro-F1 Validation à la sélection :", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Macro F1,Micro F1,Precision,Recall,Token Accuracy
1,4.539640,0.142971,0.007812,0.010989,0.100000,0.005814,0.969634
2,0.455444,0.041024,0.594441,0.751381,0.715789,0.790698,0.987762
3,0.075718,0.022589,0.659667,0.880682,0.861111,0.901163,0.994646
4,0.032319,0.023008,0.720131,0.876033,0.832461,0.924419,0.993804
5,0.027855,0.023228,0.721609,0.878453,0.836842,0.924419,0.993957


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.83it/s]


Meilleur checkpoint : C:\Users\PULSE\Documents\Master(MLAIM)\Projets personnels\ArabicLegalTextMining\artifacts\ner\augmented-20261120T212928Z-abcb8b06\checkpoint-120
Macro-F1 Validation à la sélection : 0.721608532838194


## 6. Validation du meilleur checkpoint
Les poids chargés sont ceux du meilleur Macro-F1 sur Validation.
Affichage détaillé sans ajustement des paramètres.

In [108]:
validation_prediction = trainer.predict(model_data["validation"], metric_key_prefix="validation")
validation_report = entity_report((validation_prediction.predictions, validation_prediction.label_ids))
show_report(validation_report, "Validation — meilleur checkpoint")
require(math.isclose(validation_report["macro"]["f1"], float(trainer.state.best_metric),
                     rel_tol=1e-6, abs_tol=1e-8),
        "Le score rechargé ne correspond pas au checkpoint sélectionné")

Validation — meilleur checkpoint
Entité      Precision     Recall         F1   Support
PER            0.8167     0.8909     0.8522        55
COURT          0.9518     0.9875     0.9693        80
ORG            0.6667     0.9375     0.7792        32
LOC            0.5000     0.2000     0.2857         5
Macro-F1: 0.7216 | Micro-F1: 0.8785


In [119]:
BEST_MODEL_DIR = RUN_DIR / "best_model"
trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))
trainer.save_state()
trainer.save_metrics("train", train_result.metrics)
trainer.save_metrics("validation", validation_prediction.metrics)
save_json(RUN_DIR / "validation_report.json", validation_report)
parameters.update(best_checkpoint=str(best_checkpoint),
                  best_validation_macro_f1=float(trainer.state.best_metric))
save_json(RUN_DIR / "parameters.json", parameters)
save_json(BEST_MODEL_DIR / "ner_metadata.json", {
    "base_model": MODEL_NAME, "max_length": MAX_LENGTH,
    "label2id": label2id, "id2label": id2label,
    "preprocessing": "original_text_no_external_normalization",
    "windowing": "non_overlapping_entity_preserving",
    "training_tokenization": "annotation_boundary_fragments_with_global_offsets",
    "special_and_padding_label": -100,
    "annotation_quality": "silver_unreviewed_plus_synthetic_template",
    "selection_split": "validation", "selection_metric": "eval_macro_f1",
})
print("Modèle et tokenizer sauvegardés :", BEST_MODEL_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.44it/s]

Modèle et tokenizer sauvegardés : C:\Users\PULSE\Documents\Master(MLAIM)\Projets personnels\ArabicLegalTextMining\artifacts\ner\augmented-20261120T212928Z-abcb8b06\best_model


## 7. Évaluation finale sur Test
Premier chargement de Test, après sélection et sauvegarde du modèle.
Mêmes contrôles, tokenizer, fenêtres et métriques. Un doublon avec Train/Validation
bloque l'évaluation. Aucun score Test ne sert à choisir un checkpoint ou un paramètre.

In [120]:
test_records = load_jsonl(PATHS["test"])
test_hash = sha256(PATHS["test"])
# Copies : réexécuter cette cellule ne crée pas de faux doublons.
test_audit = audit_records(test_records, "test", dict(seen_ids), dict(seen_texts))
show_audit(test_audit)
test_windows = prepare_windows(test_records, "test")

test | documents: 20 | {'PER': 107, 'COURT': 136, 'ORG': 41, 'LOC': 4}
Écarts BIO/spans: 20
test | documents: 20 | fenêtres: 52


In [114]:
test_prediction = trainer.predict(test_windows, metric_key_prefix="test")
test_report = entity_report((test_prediction.predictions, test_prediction.label_ids))
show_report(test_report, "Test — meilleur checkpoint sélectionné sur Validation")

Test — meilleur checkpoint sélectionné sur Validation
Entité      Precision     Recall         F1   Support
PER            0.8448     0.9159     0.8789       107
COURT          0.9559     0.9559     0.9559       136
ORG            0.9111     1.0000     0.9535        41
LOC            0.7500     0.7500     0.7500         4
Macro-F1: 0.8846 | Micro-F1: 0.9236


## 8. Export final
Le nouveau dossier conserve paramètres, empreintes des données, audits, métriques
et rapports par entité. Les anciens modèles et résultats sont intacts.

In [121]:
trainer.save_metrics("test", test_prediction.metrics)
save_json(RUN_DIR / "test_report.json", test_report)
save_json(RUN_DIR / "test_data_audit.json", test_audit)
for split, report in (("validation", validation_report), ("test", test_report)):
    with (RUN_DIR / f"{split}_entities.csv").open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["entity", "precision", "recall", "f1", "support"])
        writer.writeheader()
        writer.writerows({"entity": kind, **row} for kind, row in report["per_entity"].items())
save_json(RUN_DIR / "test_manifest.json", {
    "path": str(PATHS["test"]), "sha256": test_hash,
    "selected_checkpoint": str(best_checkpoint), "selection_split": "validation",
    "selection_metric": "eval_macro_f1", "test_used_for_selection": False,
    "documents": len(test_records), "windows": len(test_windows),
})
for split, original_hash in {**source_hashes, "test": test_hash}.items():
    require(sha256(PATHS[split]) == original_hash, f"{split}: fichier modifié pendant l'exécution")
print("Expérience sauvegardée :", RUN_DIR)
print("Modèle portable :", BEST_MODEL_DIR)

Expérience sauvegardée : C:\Users\PULSE\Documents\Master(MLAIM)\Projets personnels\ArabicLegalTextMining\artifacts\ner\augmented-20261120T212928Z-abcb8b06
Modèle portable : C:\Users\PULSE\Documents\Master(MLAIM)\Projets personnels\ArabicLegalTextMining\artifacts\ner\augmented-20261120T212928Z-abcb8b06\best_model
